# Seleccion de Variables y EDA — Modelo de Fuga NBC
**Etapa: calidad de datos, particion temporal, encoding y seleccion de features**

1. Config e imports (AWS) · 2. FeatureSelectionConfig (exclusiones de leakage) · 3. Carga del universo (Athena) · 4. Calidad de datos y target · 5. Clasificacion e imputacion · 6. Particion temporal (OOT) · 7. Target encoding (ciiu_val) out-of-fold · 8. Rankings (IV + correlacion + RF) · 9. Multicolinealidad y seleccion final · 10. WOE binning y datasets finales

_Variables de ubigeo/ciudad excluidas por decision de negocio._

In [ ]:
# Instalacion consolidada de dependencias
%pip install -q awswrangler==3.13.0 jupyterlab_execute_time scorecardpy seaborn scikit-learn boto3 catboost interpret optbinning xgboost==2.1.3 lightgbm statsmodels joblib optuna

In [ ]:
# ==============================================================================
# CONFIGURACIÓN Y LIBRERÍAS
# ==============================================================================

# Nativos
from dateutil.relativedelta import relativedelta
from time import gmtime, strftime
from datetime import datetime 
import random as rn
import joblib
import time
import json
import sys
import os
import gc

# Nube
import awswrangler as wr
import boto3

# Cálculo
import pandas as pd
import numpy as np
import scipy
from scipy.stats import chi2_contingency

# Gráfico
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set(style="whitegrid")

# ML
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
import scorecardpy as sc

# Configuración
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Seeds para reproducibilidad
SEED = 123
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
rn.seed(SEED)


#from utils_campania import *
from feature_selection_v1 import *

In [ ]:

import awswrangler as wr

#calculo
import pandas as pd
import numpy as np
import scipy

import boto3
from config import AWS_CONFIG, ATHENA_CONFIG
#nube

import importlib, config
importlib.reload(config)
from config import AWS_CONFIG, ATHENA_CONFIG

session = boto3.Session(**AWS_CONFIG)

sts = session.client('sts', region_name=AWS_CONFIG.get('region_name'))
cuenta = sts.get_caller_identity()['Account']

region = session.region_name or AWS_CONFIG.get('region_name')
print(cuenta, region)

# Cliente Glue con credenciales explícitas
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CONFIG["aws_access_key_id"],
    aws_secret_access_key=AWS_CONFIG["aws_secret_access_key"],
    aws_session_token=AWS_CONFIG.get("aws_session_token"),
    region_name=AWS_CONFIG["region_name"]
)

In [ ]:
# =============================================================================
# 1. CONFIGURACIÓN DEL PROYECTO
# =============================================================================

# Crear configuración
config = FeatureSelectionConfig(
    target_column='target',
    time_column='periodo_campania',
    id_column='numeroruc',
    columnas_excluir=[
        'numeroruc',
        'num_ruc' ,
        'periodo_campania', 
        'target'
    ],
    n_test_periodos=2,
    n_valid_periodos=2,
    umbral_correlacion=0.95,
    umbral_iv_min=0.001,
    umbral_iv_max=0.9,
    top_n_variables=100,
    woe_method='tree',
    random_state=123
)


## **UNIVERSO LEADS CARTERA**

In [ ]:
# ══════════════════════════════════════════════════════════
# PASO 1: HM_UNIVERSO_PXC_IZIPAY_RANGOS
# ══════════════════════════════════════════════════════════

sql_paso1 = """
      SELECT *  FROM disc_comercial.hm_kbr_base_modelo_fuga
"""

print("✓ Query PASO 1 construida (HM_UNIVERSO_PXC_IZIPAY_RANGOS)")
print("  ✓ Agregación corregida: 1 fila por (codmes, numeroruc) usando MAX() en dimensiones")

In [ ]:
# Ejecutar sql_paso1 para construir df_paso1
print("Ejecutando PASO 1: hm_kbr_base_modelo_fuga...")

try:
    df = wr.athena.read_sql_query(
        sql_paso1,
        database=ATHENA_CONFIG['database'],
        s3_output=ATHENA_CONFIG['s3_output'],
        workgroup=ATHENA_CONFIG['workgroup'],
        ctas_approach=False,
        boto3_session=session
    )

    print(f"✓ df_paso1 creado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print(f"  Períodos: {df['periodo_campania'].min()} → {df['codmes'].max()}")
    print(f"  RUCs únicos: {df['numeroruc'].nunique():,}")
    display(df.head())

except Exception as e:
    print(f"✗ Error al crear df_paso1 desde sql_paso1: {str(e)}")

In [ ]:
df

In [ ]:
# =============================================================================
# 3. ANÁLISIS INICIAL
# =============================================================================

print_seccion("ANÁLISIS INICIAL")

# Variables categóricas que NO deben convertirse a numérico
vars_categoricas_excluir = ['ciiu_val']

# Convertir object a numeric cuando aplique (excepto columnas excluidas y categóricas)
for col in df.select_dtypes(include=['object']).columns:
    if col not in config.COLUMNAS_EXCLUIR and col not in vars_categoricas_excluir:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# -----------------------------------------------------------------------------
# 3.1 Calidad de datos: ceros y nulos (función existente)
# -----------------------------------------------------------------------------
resultado_calidad = analisis_ceros_nulos(df)
print("\nTop 10 variables con más nulos:")
print(resultado_calidad.head(10)[['tipo', 'n_nulos', 'pct_nulos']])

# -----------------------------------------------------------------------------
# 3.2 Lista completa de variables con estadísticas
#      Incluye: cantidad de valores, min, max, media, mediana, percentiles, etc.
# -----------------------------------------------------------------------------
resumen_variables = []
n_filas = len(df)

for col in df.columns:
    s = df[col]
    n_nulos = int(s.isna().sum())
    n_no_nulos = int(s.notna().sum())
    n_unicos = int(s.nunique(dropna=True))

    fila = {
        'variable': col,
        'tipo_dato': str(s.dtype),
        'n_registros': n_filas,
        'n_no_nulos': n_no_nulos,
        'n_nulos': n_nulos,
        'pct_nulos': round(n_nulos / n_filas * 100, 2),
        'n_unicos': n_unicos,
        'pct_unicos': round(n_unicos / n_filas * 100, 2),
    }

    if pd.api.types.is_numeric_dtype(s):
        s_num = pd.to_numeric(s, errors='coerce')
        fila.update({
            'n_ceros': int((s_num == 0).sum()),
            'pct_ceros': round((s_num == 0).sum() / n_filas * 100, 2),
            'mean': round(s_num.mean(), 6) if s_num.notna().any() else np.nan,
            'mediana': round(s_num.median(), 6) if s_num.notna().any() else np.nan,
            'std': round(s_num.std(), 6) if s_num.notna().any() else np.nan,
            'min': round(s_num.min(), 6) if s_num.notna().any() else np.nan,
            'p01': round(s_num.quantile(0.01), 6) if s_num.notna().any() else np.nan,
            'p05': round(s_num.quantile(0.05), 6) if s_num.notna().any() else np.nan,
            'p25': round(s_num.quantile(0.25), 6) if s_num.notna().any() else np.nan,
            'p75': round(s_num.quantile(0.75), 6) if s_num.notna().any() else np.nan,
            'p95': round(s_num.quantile(0.95), 6) if s_num.notna().any() else np.nan,
            'p99': round(s_num.quantile(0.99), 6) if s_num.notna().any() else np.nan,
            'max': round(s_num.max(), 6) if s_num.notna().any() else np.nan,
        })
    else:
        # Para variables no numéricas guardamos moda/frecuencia como referencia
        vc = s.value_counts(dropna=True)
        fila.update({
            'n_ceros': np.nan,
            'pct_ceros': np.nan,
            'mean': np.nan,
            'mediana': np.nan,
            'std': np.nan,
            'min': np.nan,
            'p01': np.nan,
            'p05': np.nan,
            'p25': np.nan,
            'p75': np.nan,
            'p95': np.nan,
            'p99': np.nan,
            'max': np.nan,
            'moda': vc.index[0] if len(vc) > 0 else np.nan,
            'freq_moda': int(vc.iloc[0]) if len(vc) > 0 else 0,
        })

    resumen_variables.append(fila)

resumen_variables_df = pd.DataFrame(resumen_variables)

# Orden de columnas final
columnas_finales = [
    'variable', 'tipo_dato', 'n_registros', 'n_no_nulos', 'n_nulos', 'pct_nulos',
    'n_unicos', 'pct_unicos', 'n_ceros', 'pct_ceros',
    'min', 'p01', 'p05', 'p25', 'mediana', 'mean', 'p75', 'p95', 'p99', 'max', 'std',
    'moda', 'freq_moda'
 ]
columnas_finales = [c for c in columnas_finales if c in resumen_variables_df.columns]
resumen_variables_df = resumen_variables_df[columnas_finales]

print(f"\nTotal de variables analizadas: {resumen_variables_df.shape[0]}")
display(resumen_variables_df)

# Guardar reporte completo CSV
os.makedirs('outputs', exist_ok=True)
ruta_resumen = 'outputs/resumen_completo_variables_eda.csv'
resumen_variables_df.to_csv(ruta_resumen, index=False, encoding='utf-8-sig')
print(f"\nReporte CSV guardado en: {ruta_resumen}")

# -----------------------------------------------------------------------------
# 3.3 Rankings de calidad (% nulos y % ceros)
# -----------------------------------------------------------------------------
columnas_ranking = [
    'variable', 'tipo_dato', 'n_nulos', 'pct_nulos', 'n_ceros', 'pct_ceros',
    'n_unicos', 'pct_unicos'
 ]
columnas_ranking = [c for c in columnas_ranking if c in resumen_variables_df.columns]

ranking_calidad = resumen_variables_df[columnas_ranking].copy()
ranking_nulos = ranking_calidad.sort_values(['pct_nulos', 'n_nulos'], ascending=False).reset_index(drop=True)
ranking_ceros = ranking_calidad[ranking_calidad['pct_ceros'].notna()].sort_values(
    ['pct_ceros', 'n_ceros'], ascending=False
).reset_index(drop=True)

print("\nTop 20 variables con mayor % de nulos:")
display(ranking_nulos.head(20))

print("\nTop 20 variables con mayor % de ceros (solo numéricas):")
display(ranking_ceros.head(20))

# Guardar rankings en CSV
ruta_ranking_nulos = 'outputs/ranking_pct_nulos.csv'
ruta_ranking_ceros = 'outputs/ranking_pct_ceros.csv'
ranking_nulos.to_csv(ruta_ranking_nulos, index=False, encoding='utf-8-sig')
ranking_ceros.to_csv(ruta_ranking_ceros, index=False, encoding='utf-8-sig')
print(f"\nRanking nulos guardado en: {ruta_ranking_nulos}")
print(f"Ranking ceros guardado en: {ruta_ranking_ceros}")

# -----------------------------------------------------------------------------
# 3.4 Reporte HTML de calidad de datos
# -----------------------------------------------------------------------------
import io
import base64

def clasificar_calidad(pct_nulos, pct_ceros):
    pct_ceros = 0 if pd.isna(pct_ceros) else pct_ceros
    if pct_nulos >= 40 or pct_ceros >= 95:
        return 'CRITICO'
    if pct_nulos >= 20 or pct_ceros >= 80:
        return 'ALTO RIESGO'
    if pct_nulos >= 10 or pct_ceros >= 60:
        return 'MONITOREAR'
    return 'OK'

def fig_to_base64(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', bbox_inches='tight', dpi=120)
    buffer.seek(0)
    img_b64 = base64.b64encode(buffer.read()).decode('utf-8')
    plt.close(fig)
    return img_b64

reporte_html_df = ranking_calidad.copy()
reporte_html_df['pct_ceros'] = reporte_html_df['pct_ceros'].fillna(0)
reporte_html_df['clasificacion_calidad'] = reporte_html_df.apply(
    lambda x: clasificar_calidad(x['pct_nulos'], x['pct_ceros']), axis=1
)

# Gráfico Top 20 nulos
top20_nulos_plot = ranking_nulos.head(20).sort_values('pct_nulos', ascending=True)
fig1, ax1 = plt.subplots(figsize=(10, 7))
ax1.barh(top20_nulos_plot['variable'], top20_nulos_plot['pct_nulos'], color='#d62728', alpha=0.85)
ax1.set_title('Top 20 variables por % de nulos')
ax1.set_xlabel('% nulos')
img_nulos = fig_to_base64(fig1)

# Gráfico Top 20 ceros
top20_ceros_plot = ranking_ceros.head(20).sort_values('pct_ceros', ascending=True)
fig2, ax2 = plt.subplots(figsize=(10, 7))
ax2.barh(top20_ceros_plot['variable'], top20_ceros_plot['pct_ceros'], color='#1f77b4', alpha=0.85)
ax2.set_title('Top 20 variables por % de ceros')
ax2.set_xlabel('% ceros')
img_ceros = fig_to_base64(fig2)

# Tablas para HTML
tabla_resumen_html = reporte_html_df.sort_values(['pct_nulos', 'pct_ceros'], ascending=False).to_html(index=False)
tabla_top_nulos_html = ranking_nulos.head(30).to_html(index=False)
tabla_top_ceros_html = ranking_ceros.head(30).to_html(index=False)

n_ok = int((reporte_html_df['clasificacion_calidad'] == 'OK').sum())
n_monitorear = int((reporte_html_df['clasificacion_calidad'] == 'MONITOREAR').sum())
n_alto = int((reporte_html_df['clasificacion_calidad'] == 'ALTO RIESGO').sum())
n_critico = int((reporte_html_df['clasificacion_calidad'] == 'CRITICO').sum())

html_content = f"""
<!DOCTYPE html>
<html lang='es'>
<head>
    <meta charset='UTF-8'>
    <title>Reporte Calidad de Variables</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 24px; color: #1f2937; }}
        h1, h2 {{ color: #0f172a; }}
        .cards {{ display: flex; gap: 12px; flex-wrap: wrap; margin-bottom: 16px; }}
        .card {{ border: 1px solid #e5e7eb; border-radius: 8px; padding: 12px 14px; min-width: 180px; }}
        .ok {{ background: #ecfdf5; }}
        .mon {{ background: #fff7ed; }}
        .alto {{ background: #fef2f2; }}
        .cri {{ background: #fee2e2; }}
        table {{ border-collapse: collapse; width: 100%; font-size: 12px; margin-bottom: 20px; }}
        th, td {{ border: 1px solid #e5e7eb; padding: 6px 8px; text-align: left; }}
        th {{ background: #f8fafc; position: sticky; top: 0; }}
        .img-wrap {{ margin: 12px 0 20px 0; }}
        img {{ max-width: 100%; height: auto; border: 1px solid #e5e7eb; border-radius: 6px; }}
        .small {{ color: #6b7280; font-size: 12px; }}
    </style>
</head>
<body>
    <h1>Reporte de Calidad de Variables</h1>
    <p class='small'>Generado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>

    <div class='cards'>
        <div class='card ok'><b>OK</b><br>{n_ok}</div>
        <div class='card mon'><b>Monitorear</b><br>{n_monitorear}</div>
        <div class='card alto'><b>Alto Riesgo</b><br>{n_alto}</div>
        <div class='card cri'><b>Critico</b><br>{n_critico}</div>
    </div>

    <h2>Top variables por % de nulos</h2>
    <div class='img-wrap'><img src='data:image/png;base64,{img_nulos}' alt='Top nulos'></div>
    {tabla_top_nulos_html}

    <h2>Top variables por % de ceros</h2>
    <div class='img-wrap'><img src='data:image/png;base64,{img_ceros}' alt='Top ceros'></div>
    {tabla_top_ceros_html}

    <h2>Detalle completo de calidad</h2>
    {tabla_resumen_html}
</body>
</html>
"""

ruta_html_calidad = 'outputs/reporte_calidad_variables.html'
with open(ruta_html_calidad, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\nReporte HTML guardado en: {ruta_html_calidad}")

# -----------------------------------------------------------------------------
# 3.5 Distribución del target por período
# -----------------------------------------------------------------------------
if config.TIME_COLUMN:
    target_mensual = df.groupby([config.TIME_COLUMN]).agg(
        total=(config.TARGET_COLUMN, 'count'),
        positivos=(config.TARGET_COLUMN, 'sum'),
    ).reset_index()
    target_mensual['tasa'] = (target_mensual['positivos'] / target_mensual['total'] * 100).round(2)
    print(f"\nDistribución del target por período:")
    print(target_mensual)

## 4. CLASIFICACIÓN E IMPUTACIÓN

In [ ]:
# Clasificar variables
df_procesado, var_num, var_cat = clasifica_variables(
    df,
    target_col=config.TARGET_COLUMN,
    variables_excluir=config.COLUMNAS_EXCLUIR,
    verbose=True
)

# Imputar
df_imputado, resumen_imputacion = imputar_valores(
    df_procesado,
    var_num,
    var_cat,
    variables_excluir=[config.TIME_COLUMN, config.ID_COLUMN],
    verbose=True
)

del df, df_procesado
gc.collect()


In [ ]:
resumen_imputacion

## 4.1 Diagnóstico de Feature Selection por Calidad de Datos

Resumen de restricciones por `% nulos` y `% ceros` antes del pipeline de selección.

In [ ]:
# =============================================================================
# DIAGNÓSTICO DE FEATURE SELECTION POR CALIDAD DE DATOS
# =============================================================================

# Umbrales propuestos (ajustables)
UMBRAL_NULOS_RESTRICCION = 70.0
UMBRAL_CEROS_RESTRICCION = 90.0
UMBRAL_NULOS_ALERTA = 20.0
UMBRAL_CEROS_ALERTA = 80.0

# Base para diagnóstico
diag_fs = resumen_variables_df[[
    'variable', 'tipo_dato', 'pct_nulos', 'pct_ceros', 'n_unicos', 'pct_unicos'
]].copy()

diag_fs['pct_ceros'] = diag_fs['pct_ceros'].fillna(0)
diag_fs['pct_nulos'] = diag_fs['pct_nulos'].fillna(0)
diag_fs['es_excluida_config'] = diag_fs['variable'].isin(config.COLUMNAS_EXCLUIR)

# Semáforo de calidad
diag_fs['estado_calidad'] = np.select(
    [
        (diag_fs['pct_nulos'] >= UMBRAL_NULOS_RESTRICCION) | (diag_fs['pct_ceros'] >= UMBRAL_CEROS_RESTRICCION),
        (diag_fs['pct_nulos'] >= UMBRAL_NULOS_ALERTA) | (diag_fs['pct_ceros'] >= UMBRAL_CEROS_ALERTA),
    ],
    [
        'RESTRINGIR',
        'ALERTA',
    ],
    default='OK'
 )

# Restricción efectiva para feature selection (sin tocar columnas excluidas por config)
diag_fs['restringida_fs'] = (
    (diag_fs['estado_calidad'] == 'RESTRINGIR') &
    (~diag_fs['es_excluida_config'])
)

# Candidatas para modelado por calidad
diag_fs['candidata_por_calidad'] = (
    ~diag_fs['restringida_fs'] & ~diag_fs['es_excluida_config']
)

# Resumen ejecutivo
total_vars = len(diag_fs)
total_excluidas_config = int(diag_fs['es_excluida_config'].sum())
total_restringidas = int(diag_fs['restringida_fs'].sum())
total_alerta = int((diag_fs['estado_calidad'] == 'ALERTA').sum())
total_ok = int((diag_fs['estado_calidad'] == 'OK').sum())
total_candidatas = int(diag_fs['candidata_por_calidad'].sum())

print_seccion('RESUMEN DE RESTRICCIÓN POR NULOS/CEROS')
print(f"Total variables analizadas: {total_vars}")
print(f"Excluidas por config: {total_excluidas_config}")
print(f"Restringidas por calidad: {total_restringidas}")
print(f"En alerta: {total_alerta}")
print(f"OK: {total_ok}")
print(f"Candidatas para FS (calidad): {total_candidatas}")

print('\nTop 20 variables restringidas (peor calidad):')
display(
    diag_fs[diag_fs['restringida_fs']]
    .sort_values(['pct_nulos', 'pct_ceros'], ascending=False)
    .head(20)
    [['variable', 'tipo_dato', 'pct_nulos', 'pct_ceros', 'estado_calidad']]
)

print('\nTop 20 variables en alerta:')
display(
    diag_fs[(diag_fs['estado_calidad'] == 'ALERTA') & (~diag_fs['es_excluida_config'])]
    .sort_values(['pct_nulos', 'pct_ceros'], ascending=False)
    .head(20)
    [['variable', 'tipo_dato', 'pct_nulos', 'pct_ceros', 'estado_calidad']]
)

# Exportar diagnóstico
ruta_diag_fs = 'outputs/diagnostico_restricciones_feature_selection.csv'
diag_fs.to_csv(ruta_diag_fs, index=False, encoding='utf-8-sig')
print(f"\nDiagnóstico exportado en: {ruta_diag_fs}")

# Si ya existe ranking de variables, cruzar calidad vs importancia
if 'df_rankings' in globals() and isinstance(df_rankings, pd.DataFrame) and len(df_rankings) > 0:
    cruce_fs = df_rankings[['variable', 'iv', 'ranking']].merge(
        diag_fs[['variable', 'estado_calidad', 'restringida_fs', 'es_excluida_config']],
        on='variable', how='left'
    )

    print('\nCruce calidad vs importancia (Top IV restringidas):')
    display(
        cruce_fs[cruce_fs['restringida_fs'] == True]
        .sort_values('iv', ascending=False)
        .head(20)
    )

    ruta_cruce_fs = 'outputs/cruce_calidad_vs_ranking.csv'
    cruce_fs.to_csv(ruta_cruce_fs, index=False, encoding='utf-8-sig')
    print(f"Cruce calidad/ranking exportado en: {ruta_cruce_fs}")
else:
    print('\nAún no existe df_rankings en memoria. Ejecuta la sección de rankings para ver el cruce calidad vs importancia.')

## 5. PARTICIÓN TEMPORAL

In [ ]:
if config.TIME_COLUMN:
    train, valid, test = particionar_temporal(
        df_imputado,
        col_periodo=config.TIME_COLUMN,
        target=config.TARGET_COLUMN,
        n_test=config.N_TEST_PERIODOS,
        n_valid=config.N_VALID_PERIODOS,
        verbose=True
    )
else:
    # Partición aleatoria si no hay tiempo
    from sklearn.model_selection import train_test_split
    
    train, temp = train_test_split(df_imputado, test_size=0.3, random_state=config.RANDOM_STATE)
    valid, test = train_test_split(temp, test_size=0.5, random_state=config.RANDOM_STATE)
    
    print_seccion("PARTICIÓN ALEATORIA")
    print(f"\n Train: {len(train):,}")
    print(f" Valid: {len(valid):,}")
    print(f" Test: {len(test):,}")

# Separar X e y
X_train, y_train, X_valid, y_valid, X_test, y_test = separar_X_y(
    train, valid, test,
    target_col=config.TARGET_COLUMN,
    columnas_excluir=config.COLUMNAS_EXCLUIR,
    verbose=True
)

#del train, valid, test
gc.collect()

In [ ]:
# =============================================================================
# CONVERSIÓN DE VARIABLES NUMÉRICAS A CATEGÓRICAS
# =============================================================================

print_seccion("CONVERSIÓN DE VARIABLES PARA TARGET ENCODING")

# Variables que son numéricas pero representan categorías
vars_numericas_a_categoricas = ['tip_contribuyente_val']

# Convertir en los 3 datasets
for var in vars_numericas_a_categoricas:
    if var in X_train.columns:
        X_train[var] = X_train[var].astype(str)
        X_valid[var] = X_valid[var].astype(str)
        X_test[var] = X_test[var].astype(str)
        
        print(f" ✓ {var} convertida a string")
        print(f"   Valores únicos: {X_train[var].nunique()}")
        print(f"   Ejemplos: {X_train[var].unique()[:10].tolist()}")

In [ ]:
# =============================================================================
# TARGET ENCODING (RECIBE Y DEVUELVE DATASETS COMPLETOS)
# =============================================================================

vars_categoricas_para_encoding = [
    'ciiu_val'
]

# APLICAR TARGET ENCODING
train_enc, valid_enc, test_enc, encoding_maps = aplicar_target_encoding_a_datasets(
    train=train,                 
    valid=valid,                   
    test=test,                  
    target_col=config.TARGET_COLUMN,
    vars_categoricas=vars_categoricas_para_encoding,
    smoothing=20,
    min_freq=0.01,
    n_folds=5,
    verbose=True
)

In [ ]:
import joblib, os
os.makedirs('outputs', exist_ok=True)
joblib.dump(encoding_maps, 'outputs/encoding_maps.pkl')
print(f"✓ encoding_maps guardado: {len(encoding_maps)} variables")
for var, info in encoding_maps.items():
    print(f"  • {var}: {info['n_categorias_final']} categorías")

In [ ]:
# =============================================================================
# 3. LIMPIEZA INICIAL (ANTES DE LA SELECCIÓN)
# =============================================================================

print_seccion("LIMPIEZA INICIAL DE DATASETS")

# Columnas a mantener (TODO menos las categóricas originales)
columnas_mantener_limpieza = [
    config.TARGET_COLUMN,
    config.TIME_COLUMN,
    config.ID_COLUMN
]

# Limpiar (elimina categóricas originales, columnas de periodo auxiliares, etc.)
train_clean = limpia_dataset_final(
    df=train_enc,
    target_col=config.TARGET_COLUMN,
    time_col=config.TIME_COLUMN,
    id_col=config.ID_COLUMN,
    columnas_mantener=columnas_mantener_limpieza,
    verbose=True
)

valid_clean = limpia_dataset_final(
    df=valid_enc,
    target_col=config.TARGET_COLUMN,
    time_col=config.TIME_COLUMN,
    id_col=config.ID_COLUMN,
    columnas_mantener=columnas_mantener_limpieza,
    verbose=False
)

test_clean = limpia_dataset_final(
    df=test_enc,
    target_col=config.TARGET_COLUMN,
    time_col=config.TIME_COLUMN,
    id_col=config.ID_COLUMN,
    columnas_mantener=columnas_mantener_limpieza,
    verbose=False
)

print(f"\n ✓ Datasets limpios creados:")
print(f"   Train: {train_clean.shape}")
print(f"   Valid: {valid_clean.shape}")
print(f"   Test: {test_clean.shape}")

# =============================================================================
# 4. SEPARAR X e Y (PARA EL PIPELINE DE SELECCIÓN)
# =============================================================================

print_seccion("SEPARACIÓN X/Y PARA PIPELINE DE SELECCIÓN")

# Columnas a excluir de X
columnas_excluir_features = [
    config.TARGET_COLUMN,
    config.TIME_COLUMN,
    config.ID_COLUMN,
    'periodo_ejecucion',
    'acepta_campana',
    'monto_desembolsado',
    'monto_ofertado',
    'tasa_regular',
    'flg_cartera',
    'desembolsado',
    'flg_monto_desembolsos_mayor150k',
    'ciiu_val',
]

# Features (todas las columnas numéricas + _encoded)
feature_cols = [col for col in train_clean.columns if col not in columnas_excluir_features]

X_train = train_clean[feature_cols].copy()
X_valid = valid_clean[feature_cols].copy()
X_test = test_clean[feature_cols].copy()

y_train = train_clean[config.TARGET_COLUMN].copy()
y_valid = valid_clean[config.TARGET_COLUMN].copy()
y_test = test_clean[config.TARGET_COLUMN].copy()

print(f"\n Features (X) para selección:")
print(f"   X_train: {X_train.shape}")
print(f"   X_valid: {X_valid.shape}")
print(f"   X_test: {X_test.shape}")

print(f"\n Target (y):")
print(f"   y_train: {y_train.shape} | Tasa: {y_train.mean():.4f}")
print(f"   y_valid: {y_valid.shape} | Tasa: {y_valid.mean():.4f}")
print(f"   y_test: {y_test.shape} | Tasa: {y_test.mean():.4f}")

# Verificar que no hay categóricas en X
cols_categoricas = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
if len(cols_categoricas) > 0:
    print(f"\n  ADVERTENCIA: Variables categóricas en X:")
    for col in cols_categoricas:
        print(f"   • {col}")
else:
    print(f"\n ✓ X solo contiene variables numéricas")

# Verificar nulos
nulos = X_train.isnull().sum().sum()
print(f"\n Nulos en X_train: {nulos}")

In [ ]:
# Guardar los conjuntos de datos transformados sin mes ni target
X_train.to_csv('train.csv', index=False)
X_valid.to_csv('valid.csv', index=False)
X_test.to_csv('test.csv', index=False)


# Guardar los conjuntos de datos transformados pero conservando mes y target

train_clean.to_csv('final_train.csv', index=False)
valid_clean.to_csv('final_valid.csv', index=False)
test_clean.to_csv('final_test.csv', index=False)


In [ ]:
train_clean

In [ ]:
X_train.to_csv('train.csv', index=False)


In [ ]:
# =============================================================================
# 6. GENERACIÓN DE RANKINGS
# =============================================================================

# Obtener lista de todas las features (incluyendo _encoded)
feature_cols_con_encoding = [col for col in X_train.columns 
                             if col not in config.COLUMNAS_EXCLUIR]

print(f"\n Total features (con encoding): {len(feature_cols_con_encoding)}")

# Continuar con el pipeline normal (rankings, multicolinealidad, etc.)
df_rankings, top_vars = generar_ranking_variables(
    X_train=X_train[feature_cols_con_encoding],
    y_train=y_train,
    top_n=100,
    verbose=True
)

# -----------------------------------------------------------------------------
# Diagnóstico de signo de correlación (Spearman)
# Nota: correlación negativa NO es mala; solo indica relación inversa con target.
# El ranking ya usa valor absoluto de correlación.
# -----------------------------------------------------------------------------
df_rankings['correlation_abs'] = df_rankings['correlation_spearman'].abs()

n_pos = int((df_rankings['correlation_spearman'] > 0).sum())
n_neg = int((df_rankings['correlation_spearman'] < 0).sum())
n_zero = int((df_rankings['correlation_spearman'] == 0).sum())

print("\nResumen signo de correlación Spearman:")
print(f"  Positivas: {n_pos}")
print(f"  Negativas: {n_neg}")
print(f"  Cero: {n_zero}")

print("\nTop 20 por |correlación| (sin importar signo):")
display(
    df_rankings[['variable', 'iv', 'max_prob', 'correlation_spearman', 'correlation_abs']]
    .sort_values('correlation_abs', ascending=False)
    .head(20)
)


In [ ]:
X_train

In [ ]:
# =============================================================================
# 6.1 IMPORTANCIA POR RANDOM FOREST
# =============================================================================

print_seccion("IMPORTANCIA POR RANDOM FOREST")

from sklearn.ensemble import RandomForestClassifier

# Entrenar RF rápido
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=50,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)

# Usar las mismas features del ranking
X_rf = X_train[feature_cols_con_encoding].copy()
rf.fit(X_rf, y_train)

# Extraer importancias
df_rf_importance = pd.DataFrame({
    'variable': feature_cols_con_encoding,
    'rf_importance': rf.feature_importances_
}).sort_values('rf_importance', ascending=False).reset_index(drop=True)

df_rf_importance['rf_ranking'] = range(1, len(df_rf_importance) + 1)

print(f"\nTop 30 variables por importancia RF:")
display(df_rf_importance.head(30))

# AUC del RF en train y validación
from sklearn.metrics import roc_auc_score

y_train_pred = rf.predict_proba(X_rf)[:, 1]
auc_train = roc_auc_score(y_train, y_train_pred)

X_valid_rf = X_valid[feature_cols_con_encoding].copy()
y_valid_pred = rf.predict_proba(X_valid_rf)[:, 1]
auc_valid = roc_auc_score(y_valid, y_valid_pred)

print(f"\nAUC RF (todas las variables):")
print(f"  Train: {auc_train:.4f}")
print(f"  Valid: {auc_valid:.4f}")

# Merge con rankings existentes (IV + correlación + RF)
df_rankings = df_rankings.merge(
    df_rf_importance[['variable', 'rf_importance', 'rf_ranking']],
    on='variable',
    how='left'
)

# Ranking combinado (promedio de rankings)
df_rankings['ranking_combinado'] = (
    df_rankings['ranking'] + df_rankings['rf_ranking']
) / 2

df_rankings = df_rankings.sort_values('ranking_combinado').reset_index(drop=True)

print(f"\nTop 30 por ranking combinado (IV + Corr + RF):")
display(
    df_rankings[['variable', 'iv', 'correlation_abs', 'rf_importance', 'ranking', 'rf_ranking', 'ranking_combinado']]
    .head(30)
)

# Gráfico Top 20 importancia RF
fig, ax = plt.subplots(figsize=(10, 8))
top20_rf = df_rf_importance.head(20).sort_values('rf_importance', ascending=True)
ax.barh(top20_rf['variable'], top20_rf['rf_importance'], color='#2ca02c', alpha=0.85)
ax.set_title('Top 20 variables por importancia Random Forest')
ax.set_xlabel('Importancia (Gini)')
plt.tight_layout()
plt.show()

del rf
gc.collect()

In [ ]:
# =============================================================================
# 7. SELECCIÓN FINAL DE VARIABLES CON ANÁLISIS DETALLADO
# =============================================================================

print_seccion("FILTRADO POR INFORMATION VALUE")

# Filtrar por IV
df_rankings_filtrado = df_rankings[
    (df_rankings['iv'] >= config.UMBRAL_IV_MIN) & 
    (df_rankings['iv'] <= config.UMBRAL_IV_MAX)
].copy()

print(f"\n Variables después de filtro IV [{config.UMBRAL_IV_MIN}, {config.UMBRAL_IV_MAX}]: {len(df_rankings_filtrado)}")
print(f"\n Top 20 después de filtro:")
print(df_rankings_filtrado.head(20)[['variable', 'iv', 'max_prob', 'correlation_spearman']].to_string(index=False))


In [ ]:
# =============================================================================
# ANÁLISIS DETALLADO DE MULTICOLINEALIDAD
# =============================================================================

vars_filtradas_iv = df_rankings_filtrado['variable'].tolist()

cols_to_delete, df_correlaciones, df_resumen_multicol = analizar_multicolinealidad_detallado(
    X_train=X_train,
    y_train=y_train,
    vars_analizar=vars_filtradas_iv,
    target_col=config.TARGET_COLUMN,
    umbral_corr=config.UMBRAL_CORRELACION,
    verbose=True
)


In [ ]:
# =============================================================================
# APLICAR ELIMINACIÓN DE VARIABLES CORRELACIONADAS
# =============================================================================

print_seccion("ELIMINACIÓN DE VARIABLES CORRELACIONADAS")

if len(cols_to_delete) > 0:
    print(f"\n Eliminando {len(cols_to_delete)} variables por multicolinealidad...")
    
    vars_sin_multicol = [v for v in vars_filtradas_iv if v not in cols_to_delete]
    
    df_rankings_final = df_rankings_filtrado[
        df_rankings_filtrado['variable'].isin(vars_sin_multicol)
    ].copy()
    
    print(f" Variables restantes: {len(vars_sin_multicol)}")
else:
    print(f"\n ✓ No se encontraron variables con correlación alta")
    vars_sin_multicol = vars_filtradas_iv
    df_rankings_final = df_rankings_filtrado.copy()

In [ ]:

# =============================================================================
# 10. SELECCIÓN TOP N
# =============================================================================

top_n_final = min(config.TOP_N_VARIABLES, len(df_rankings_final))
df_rankings_final = df_rankings_final.head(top_n_final)
vars_seleccionadas = df_rankings_final['variable'].tolist()

print_seccion("VARIABLES SELECCIONADAS FINALES")
print(f"\n Total: {len(vars_seleccionadas)}")




In [ ]:
# =============================================================================
# 11. CREAR DATASETS FINALES CON VARIABLES SELECCIONADAS
# =============================================================================

print_seccion("DATASETS FINALES CON VARIABLES SELECCIONADAS")

# Columnas finales = vars_seleccionadas + columnas meta + columnas adicionales
columnas_finales = list(set(
    vars_seleccionadas +
    [config.TARGET_COLUMN, config.TIME_COLUMN, config.ID_COLUMN] +
    columnas_mantener_limpieza
))

# Filtrar solo las que existen
columnas_finales_disponibles = [col for col in columnas_finales if col in train_clean.columns]

# Datasets finales
train_final = train_clean[columnas_finales_disponibles].copy()
valid_final = valid_clean[columnas_finales_disponibles].copy()
test_final = test_clean[columnas_finales_disponibles].copy()

print(f"\n Datasets finales (completos):")
print(f"   Train: {train_final.shape}")
print(f"   Valid: {valid_final.shape}")
print(f"   Test: {test_final.shape}")

# =============================================================================
# 12. DATASETS PARA MODELADO (solo features seleccionadas)
# =============================================================================

X_train_model = train_final[vars_seleccionadas].copy()
X_valid_model = valid_final[vars_seleccionadas].copy()
X_test_model = test_final[vars_seleccionadas].copy()

y_train_final = train_final[config.TARGET_COLUMN].copy()
y_valid_final = valid_final[config.TARGET_COLUMN].copy()
y_test_final = test_final[config.TARGET_COLUMN].copy()

print(f"\n Datasets para modelado (solo features):")
print(f"   X_train: {X_train_model.shape}")
print(f"   X_valid: {X_valid_model.shape}")
print(f"   X_test: {X_test_model.shape}")

In [ ]:
# =============================================================================
# 8. WOE BINNING (OPCIONAL)
# =============================================================================

# Aplicar WOE binning
cortes, df_iv_summary = aplicar_woe_binning(
    X_train=X_train_model,
    y_train=y_train,
    vars_seleccionadas=vars_seleccionadas,
    config=config,
    verbose=True
)

# Transformar a WOE
X_train_woe, X_valid_woe, X_test_woe = transformar_woe(
    X_train=X_train_model,
    X_valid=X_valid_model,
    X_test=X_test_model,
    cortes=cortes,
    umbral_iv=0.02,
    verbose=True
)

In [ ]:
# =============================================================================
# 9. GENERAR REPORTE WOE HTML (SIN ARCHIVOS EXTERNOS)
# =============================================================================

if len(cortes) > 0:
    
    # Generar reporte HTML con gráficos embebidos en Base64
    reporte_html = generar_reporte_woe_html(
        cortes=cortes,
        df_iv_summary=df_iv_summary,
        vars_seleccionadas=vars_seleccionadas,
        output_file='outputs/reporte_woe.html',
        max_graficos=50,  # Número máximo de gráficos a incluir
        verbose=True
    )
    
    if reporte_html:
        print(f"\n ✓ Reporte HTML autocontenido: {reporte_html}")
        print(f"   Abre el archivo en tu navegador para ver todos los gráficos WOE")



In [ ]:
# =============================================================================
# 9. MOSTRAR GRÁFICOS WOE (SIN GUARDAR)
# =============================================================================

if len(cortes) > 0:
    
    # Opción 1: Mostrar TODAS las variables seleccionadas
    mostrar_graficos_woe(
        cortes=cortes,
        vars_seleccionadas=vars_seleccionadas,  # Solo variables finales (sin multicolinealidad)
        max_vars=None,  # None = todas, o un número como 20
        verbose=True
    )


In [ ]:
df_rankings_final

In [ ]:
# 2. Mostrar solo top 15 por IV
vars_top_15 = df_rankings_final.head(30)['variable'].tolist()
mostrar_graficos_woe(
    cortes=cortes,
    vars_seleccionadas=vars_top_15,
    verbose=True
)

In [ ]:
# 3. Mostrar solo las que tienen IV > 0.1
vars_iv_alto = df_iv_summary[df_iv_summary['iv_total'] > 0.1]['variable'].tolist()
mostrar_graficos_woe(
    cortes=cortes,
    vars_seleccionadas=vars_iv_alto,
    verbose=True
)

In [ ]:
# ==============================================================================
# CALCULAR WOE PARA VARIABLES ESPECÍFICAS
# ==============================================================================

# 1. Lista de variables que quieres ver
variables_deseadas = [




]

# 2. Verificar cuáles existen en X_train
vars_disponibles = [v for v in variables_deseadas if v in X_train.columns]
vars_faltantes = [v for v in variables_deseadas if v not in X_train.columns]

print(f"\n Variables solicitadas: {len(variables_deseadas)}")
print(f" Variables disponibles: {len(vars_disponibles)}")

if len(vars_faltantes) > 0:
    print(f" Variables NO encontradas: {len(vars_faltantes)}")
    for v in vars_faltantes:
        print(f"   ✗ {v}")

# 3. Filtrar solo numéricas
vars_numericas = X_train[vars_disponibles].select_dtypes(include=[np.number]).columns.tolist()
vars_no_numericas = [v for v in vars_disponibles if v not in vars_numericas]

if len(vars_no_numericas) > 0:
    print(f"\n Variables no numéricas (excluidas de WOE): {len(vars_no_numericas)}")
    for v in vars_no_numericas:
        print(f"   {v}")

print(f"\n Variables numéricas para WOE: {len(vars_numericas)}")

# 4. Calcular WOE binning para estas variables
if len(vars_numericas) > 0:
    try:
        import scorecardpy as sc
    except:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "scorecardpy", "--quiet"])
        import scorecardpy as sc
    
    # Preparar dataset
    df_binning_custom = X_train[vars_numericas].copy()
    df_binning_custom[config.TARGET_COLUMN] = y_train.values
    
    print(f"\n Calculando bins WOE...")
    
    try:
        # Calcular WOE
        cortes_custom = sc.woebin(
            df_binning_custom,
            y=config.TARGET_COLUMN,
            method='tree',
            bin_num_limit=8,
            positive='1',
            no_cores=1
        )
        
        print(f" ✓ Bins calculados para {len(cortes_custom)} variables")
        
        # 5. Mostrar información de IV
        print(f"\n Información de variables:")
        for var in vars_numericas:
            if var in cortes_custom:
                iv = cortes_custom[var]['total_iv'].max()
                n_bins = len(cortes_custom[var])
                print(f"   • {var:50s} | IV: {iv:.4f} | Bins: {n_bins}")
        
        # 6. Mostrar gráficos
        print(f"\n Mostrando gráficos WOE...")
        mostrar_graficos_woe(
            cortes=cortes_custom,
            vars_seleccionadas=None,  # None = todas
            verbose=True
        )
        
    except Exception as e:
        print(f"\n  Error calculando WOE: {str(e)}")
        print(f"\n Intentando método fallback...")
        
        try:
            cortes_custom = sc.woebin(
                df_binning_custom,
                y=config.TARGET_COLUMN,
                method='quantile',
                bin_num_limit=5,
                positive='1'
            )
            
            print(f" ✓ Bins calculados (quantile) para {len(cortes_custom)} variables")
            
            mostrar_graficos_woe(
                cortes=cortes_custom,
                vars_seleccionadas=None,
                verbose=True
            )
        except Exception as e2:
            print(f"  Error con fallback: {str(e2)}")

else:
    print("\n  No hay variables numéricas disponibles")

In [ ]:
# =============================================================================
# WOE ANÁLISIS - 28 VARIABLES DEL MODELO (ESTILO INTERBANK VERDE)
# =============================================================================
import matplotlib.pyplot as plt
from matplotlib import rcParams
import io, base64
plt.ioff()

def rgb_to_mpl(rgb_str):
    nums = rgb_str.replace('rgb(', '').replace(')', '').split(',')
    return tuple([int(n)/255 for n in nums])

COLOR_VERDE_IBK = rgb_to_mpl('rgb(0,208,60)')
COLOR_AZUL_IBK = rgb_to_mpl('rgb(31,69,146)')
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Poppins', 'Arial', 'Helvetica']

# ── Funciones auxiliares ──
def format_number(val, is_amount=False):
    if pd.isna(val): return "0"
    if is_amount:
        if abs(val) >= 1_000_000: return f"{val/1_000_000:.1f}M"
        elif abs(val) >= 1_000: return f"{val/1_000:.0f}K"
        else: return f"{val:.0f}"
    return f"{val:,.0f}" if abs(val) >= 1_000 else f"{val:.0f}"

def is_amount_variable(var_name):
    keywords = ['amt','saldo','deuda','monto','importe','balance','capital','credito','coloc','oferta','sldtot']
    return any(k in var_name.lower() for k in keywords)

def clasificar_iv(iv):
    if iv < 0.02: return "No predictivo"
    elif iv < 0.1: return "Débil"
    elif iv < 0.3: return "Medio"
    elif iv < 0.5: return "Fuerte"
    else: return "Muy fuerte"

def merge_small_bins(woe_df, min_pct=5.0):
    total_registros = woe_df['total_registros'].sum()
    min_registros = total_registros * (min_pct / 100)
    if woe_df['total_registros'].min() >= min_registros: return woe_df
    df_merged = woe_df.copy()
    while True:
        min_idx = df_merged['total_registros'].idxmin()
        if df_merged.loc[min_idx, 'total_registros'] >= min_registros: break
        if len(df_merged) <= 1: break
        if min_idx == 0: merge_with = min_idx + 1
        elif min_idx == len(df_merged) - 1: merge_with = min_idx - 1
        else:
            merge_with = min_idx - 1 if df_merged.loc[min_idx-1,'total_registros'] < df_merged.loc[min_idx+1,'total_registros'] else min_idx+1
        idx1, idx2 = sorted([min_idx, merge_with])
        new_row = {
            'bin': f"{df_merged.loc[idx1,'bin']}-{df_merged.loc[idx2,'bin']}",
            'total_registros': df_merged.loc[idx1,'total_registros'] + df_merged.loc[idx2,'total_registros'],
            'eventos': df_merged.loc[idx1,'eventos'] + df_merged.loc[idx2,'eventos'],
            'valor_min': min(df_merged.loc[idx1,'valor_min'], df_merged.loc[idx2,'valor_min']),
            'valor_max': max(df_merged.loc[idx1,'valor_max'], df_merged.loc[idx2,'valor_max'])
        }
        df_merged = df_merged.drop([idx1, idx2])
        df_merged = pd.concat([df_merged, pd.DataFrame([new_row])], ignore_index=True)
        df_merged = df_merged.sort_values('valor_min').reset_index(drop=True)
    return df_merged

def calculate_woe_iv_auto(df, feature_col, target_col, n_bins=5, min_bin_pct=10.0):
    try:
        temp_df = df[[feature_col, target_col]].copy().dropna()
        if len(temp_df) == 0: return None, 0, None
        n_unique = temp_df[feature_col].nunique()
        if n_unique < 2: return None, 0, None
        is_amt = is_amount_variable(feature_col)
        pct_zeros = (temp_df[feature_col] == 0).mean()
        if pct_zeros > 0.30 and n_unique > 2:
            df_zero = temp_df[temp_df[feature_col] == 0].copy()
            df_nonzero = temp_df[temp_df[feature_col] != 0].copy()
            n_nz_unique = df_nonzero[feature_col].nunique()
            target_nz_bins = max(1, min(3, n_nz_unique))
            if len(df_nonzero) > 0 and n_nz_unique >= 2:
                try: df_nonzero['bin'], _ = pd.qcut(df_nonzero[feature_col], q=target_nz_bins, duplicates='drop', labels=False, retbins=True)
                except: df_nonzero['bin'], _ = pd.cut(df_nonzero[feature_col], bins=target_nz_bins, duplicates='drop', labels=False, retbins=True)
                df_nonzero['bin'] = df_nonzero['bin'] + 1
            else: df_nonzero['bin'] = 1
            df_zero['bin'] = 0
            temp_df = pd.concat([df_zero, df_nonzero])
            binning_type = 'zero_split'
        elif n_unique <= 10:
            temp_df['bin'] = temp_df[feature_col].astype(str)
            binning_type = 'categorical'
        else:
            effective_bins = 4 if is_amt else n_bins
            try: temp_df['bin'], _ = pd.qcut(temp_df[feature_col], q=min(effective_bins, n_unique), duplicates='drop', labels=False, retbins=True); binning_type = 'quantile'
            except: temp_df['bin'], _ = pd.cut(temp_df[feature_col], bins=min(effective_bins, n_unique), duplicates='drop', labels=False, retbins=True); binning_type = 'equal_width'
        woe_df = temp_df.groupby('bin', observed=False).agg(total_registros=('bin','count'), eventos=(target_col,'sum'), valor_min=(feature_col,'min'), valor_max=(feature_col,'max')).reset_index()
        woe_df = woe_df[woe_df['total_registros'] > 0].copy()
        if len(woe_df) < 2: return None, 0, None
        if binning_type in ['quantile','equal_width']: woe_df = merge_small_bins(woe_df, min_pct=min_bin_pct)
        woe_df['no_eventos'] = woe_df['total_registros'] - woe_df['eventos']
        woe_df['tasa_pct'] = (woe_df['eventos'] / woe_df['total_registros'] * 100)
        total_ev = woe_df['eventos'].sum(); total_noev = woe_df['no_eventos'].sum()
        if total_ev == 0 or total_noev == 0: return None, 0, None
        woe_df['dist_ev'] = woe_df['eventos'] / total_ev; woe_df['dist_noev'] = woe_df['no_eventos'] / total_noev
        eps = 0.0001
        woe_df['woe'] = np.log((woe_df['dist_ev']+eps)/(woe_df['dist_noev']+eps))
        woe_df['iv'] = (woe_df['dist_ev'] - woe_df['dist_noev']) * woe_df['woe']
        woe_df['woe'] = woe_df['woe'].replace([np.inf,-np.inf],0).fillna(0)
        woe_df['iv'] = woe_df['iv'].replace([np.inf,-np.inf],0).fillna(0)
        if binning_type == 'zero_split':
            labels = []
            for _, row in woe_df.iterrows():
                if row['valor_min'] == 0 and row['valor_max'] == 0: labels.append('= 0')
                else: labels.append(f"({format_number(row['valor_min'], True)}, {format_number(row['valor_max'], True)}]")
            woe_df['bin_label'] = labels
        elif binning_type in ['quantile','equal_width']:
            if is_amt: woe_df['bin_label'] = woe_df.apply(lambda x: f"[{format_number(x['valor_min'],True)}, {format_number(x['valor_max'],True)}]", axis=1)
            else: woe_df['bin_label'] = woe_df.apply(lambda x: f"[{x['valor_min']:.1f}, {x['valor_max']:.1f}]", axis=1)
        else: woe_df['bin_label'] = woe_df['bin'].astype(str)
        woe_df['pct_registros'] = (woe_df['total_registros']/woe_df['total_registros'].sum()*100).round(1)
        resultado = woe_df[['bin_label','total_registros','pct_registros','eventos','tasa_pct','woe','iv']].copy()
        resultado.columns = ['Rango','Total','% Total','Eventos','Tasa_%','WOE','IV']
        resultado['Tasa_%'] = resultado['Tasa_%'].round(2); resultado['WOE'] = resultado['WOE'].round(4); resultado['IV'] = resultado['IV'].round(4)
        return resultado, woe_df['iv'].sum(), binning_type
    except: return None, 0, None

# ── Visualización WOE estilo Interbank ──
def plot_woe_verde(woe_df, variable_name, num_var, total_vars):
    plt.ioff()
    fig = plt.figure(figsize=(10, 15)); gs = fig.add_gridspec(2, 1, hspace=0.5); fig.patch.set_facecolor('white')
    x_pos = np.arange(len(woe_df))
    # Panel 1: Distribución + Tasa
    ax1 = fig.add_subplot(gs[0,0]); ax1.set_facecolor('white')
    bars = ax1.bar(x_pos, woe_df['Total'], width=0.7, color=COLOR_VERDE_IBK, alpha=0.85, edgecolor=COLOR_VERDE_IBK, linewidth=1.5)
    for bar, val, pct in zip(bars, woe_df['Total'], woe_df['% Total']):
        h = bar.get_height()
        if h > 0: ax1.text(bar.get_x()+bar.get_width()/2., h*0.5, f'{int(val):,}\n({pct}%)', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    ax1_tw = ax1.twinx()
    ax1_tw.plot(x_pos, woe_df['Tasa_%'], color=COLOR_AZUL_IBK, marker='o', markersize=12, linewidth=4, zorder=5)
    for x, y in zip(x_pos, woe_df['Tasa_%']):
        ax1_tw.annotate(f'{y:.1f}%', xy=(x,y), xytext=(0,15), textcoords='offset points', ha='center', fontsize=12, fontweight='bold', color=COLOR_AZUL_IBK,
                        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor=COLOR_AZUL_IBK, linewidth=2))
    tasa_prom = woe_df['Eventos'].sum()/woe_df['Total'].sum()*100
    ax1_tw.axhline(y=tasa_prom, color=COLOR_AZUL_IBK, linestyle='--', alpha=0.5, linewidth=2.5)
    ax1_tw.text(len(woe_df)-0.5, tasa_prom, f'Tasa Promedio: {tasa_prom:.2f}%', fontsize=11, fontweight='bold', color=COLOR_AZUL_IBK, ha='right', va='bottom',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=COLOR_AZUL_IBK, linewidth=1.5, alpha=0.9))
    ax1.set_xlabel('Rango', fontsize=15, fontweight='bold', color=COLOR_AZUL_IBK)
    ax1.set_ylabel('Total Registros', fontsize=15, fontweight='bold', color=COLOR_VERDE_IBK)
    ax1_tw.set_ylabel('Tasa Evento %', fontsize=15, fontweight='bold', color=COLOR_AZUL_IBK)
    ax1.set_xticks(x_pos); ax1.set_xticklabels(woe_df['Rango'], rotation=45, ha='right', fontsize=12)
    ax1.set_title('Distribución y Tasa de Evento (Target)', fontsize=16, fontweight='bold', color=COLOR_AZUL_IBK, pad=20)
    ax1.grid(True, axis='y', alpha=0.3, linestyle='--'); ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False); ax1_tw.spines['top'].set_visible(False)
    # Panel 2: WOE
    ax2 = fig.add_subplot(gs[1,0]); ax2.set_facecolor('white')
    colors_woe = [COLOR_VERDE_IBK if w >= 0 else COLOR_AZUL_IBK for w in woe_df['WOE']]
    bars2 = ax2.bar(x_pos, woe_df['WOE'], width=0.7, color=colors_woe, alpha=0.85, edgecolor='black', linewidth=1.5)
    for bar, wv, iv in zip(bars2, woe_df['WOE'], woe_df['IV']):
        h = bar.get_height()
        if abs(h) > 0.01:
            yp = h + (0.12 if h >= 0 else -0.12)
            ax2.text(bar.get_x()+bar.get_width()/2, yp, f'WOE: {wv:.3f}\nIV: {iv:.4f}', ha='center', va='bottom' if h>=0 else 'top',
                     fontsize=11, fontweight='bold', color=COLOR_AZUL_IBK, bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=COLOR_AZUL_IBK, linewidth=1.5, alpha=0.9))
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=2)
    ax2.set_xlabel('Rango', fontsize=15, fontweight='bold', color=COLOR_AZUL_IBK)
    ax2.set_ylabel('WOE', fontsize=15, fontweight='bold', color=COLOR_AZUL_IBK)
    ax2.set_xticks(x_pos); ax2.set_xticklabels(woe_df['Rango'], rotation=45, ha='right', fontsize=12)
    total_iv = woe_df['IV'].sum()
    ax2.set_title(f'Weight of Evidence (WOE) | IV Total: {total_iv:.4f} ({clasificar_iv(total_iv)})', fontsize=16, fontweight='bold', color=COLOR_AZUL_IBK, pad=20)
    ax2.grid(True, axis='y', alpha=0.3, linestyle='--'); ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
    plt.suptitle(f'[{num_var}/{total_vars}] Análisis WOE - {variable_name}', fontsize=18, fontweight='bold', color=COLOR_AZUL_IBK, y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.99]); plt.show(); plt.close('all')

def plot_iv_ranking_verde(summary_df, target_name='target'):
    plt.ioff()
    fig, ax = plt.subplots(figsize=(16, max(12, len(summary_df) * 0.45)))
    fig.patch.set_facecolor('white'); ax.set_facecolor('white')
    plot_data = summary_df.sort_values('iv', ascending=True).copy()
    color_map = {'Muy fuerte': COLOR_VERDE_IBK, 'Fuerte': (0.4,0.7,0.4), 'Medio': (1.0,0.7,0.0), 'Débil': (1.0,0.4,0.4), 'No predictivo': (0.7,0.7,0.7)}
    colors = [color_map.get(p, (0.8,0.8,0.8)) for p in plot_data['poder_predictivo']]
    bars = ax.barh(range(len(plot_data)), plot_data['iv'], color=colors, edgecolor='black', linewidth=1, alpha=0.85)
    for i, (bar, iv_val) in enumerate(zip(bars, plot_data['iv'])):
        if bar.get_width() > 0.005:
            ax.text(bar.get_width()+0.008, bar.get_y()+bar.get_height()/2, f'{iv_val:.4f}', va='center', fontsize=10, fontweight='bold', color=COLOR_AZUL_IBK)
    ax.axvline(x=0.02, color='gray', linestyle='--', alpha=0.4); ax.axvline(x=0.1, color='red', linestyle='--', alpha=0.5); ax.axvline(x=0.3, color='orange', linestyle='--', alpha=0.5)
    ax.set_yticks(range(len(plot_data))); ax.set_yticklabels(plot_data['variable'], fontsize=11)
    ax.set_xlabel('Information Value (IV)', fontsize=15, fontweight='bold', color=COLOR_AZUL_IBK)
    ax.set_title(f'Ranking Variables por IV | Target: {target_name} | {len(plot_data)} variables', fontsize=16, fontweight='bold', color=COLOR_AZUL_IBK, pad=20)
    ax.grid(axis='x', alpha=0.3, linestyle='--'); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout(); plt.show(); plt.close('all')

# ══════════════════════════════════════════════════════════════════════════════
# EJECUCIÓN: 28 VARIABLES DEL MODELO
# ══════════════════════════════════════════════════════════════════════════════

variables_modelo = [
'ciiu_val_encoded',
    'tiempo_vida_empresa',
    'flg_pj',
    'ingreso_bruto_total_rrll',
    'prm_sldtotfintrx12m',
    'prm_sldtotfintrx06m',
    'max_sldtotfinpr06m',
    'max_sldtotprmpr06m',
    'max_sldtotfinpr03m',
    'deuda_total_max_12m',
    'cant_empresas_max_12m',
    'deuda_total_var_pct_12m',
    'rrll_deuda_total',
    'rrll_cant_entidades',
    'ibk_noizi_mm6',
    'ibk_noizi_mm3',
    'ibk_noizi_tc_mm3',
    'ibk_noizi_visa_mm3',
    'ibk_noizi_max6',
    'ibk_noizi_min3',
    'cre_sldtotfintrxm03',
    'max_saldo_p_u6m',
    'rat_sldtotfinprm12',
    'tot_ratio_max_mm3',
    'tot_ratio_mm3_mm6',
    'tot_monto_mm3',
    'promedio_movil_pasivo_3m',
    'frecuencia_pasivo_mayor_10k_pct_6m',
    'recencia_pasivo_mayor_10k_meses',
    'var_pct_pasivo_3m',
    'tot_aceleracion_pos',
    'tasa_actividad_6m_pos',
    'ibk_noizi_aceleracion'
]

# Preparar df con target
df_woe_base = X_train[variables_modelo].copy()
df_woe_base[config.TARGET_COLUMN] = y_train.values

print(f"\n{'='*100}")
print(f"ANÁLISIS WOE - {len(variables_modelo)} VARIABLES DEL MODELO vs TARGET")
print(f"Saldos/Montos: estrategia zero-split (=0 | bajo | medio | alto)")
print(f"{'='*100}\n")

resumen_list = []
detalle = {}

for i, var in enumerate(variables_modelo, 1):
    print(f"[{i}/{len(variables_modelo)}] {var:45s}...", end=' ')
    woe_r, iv_t, bt = calculate_woe_iv_auto(df_woe_base, var, config.TARGET_COLUMN, n_bins=5, min_bin_pct=10.0)
    if woe_r is not None:
        detalle[var] = woe_r
        resumen_list.append({'variable': var, 'iv': iv_t, 'n_bins': len(woe_r), 'binning_type': bt, 'poder_predictivo': clasificar_iv(iv_t)})
        print(f"✓ IV={iv_t:.4f} ({clasificar_iv(iv_t)}) - {len(woe_r)} bins [{bt}]")
    else:
        print(f"✗ Sin resultado")

summary_df_woe = pd.DataFrame(resumen_list).sort_values('iv', ascending=False).reset_index(drop=True)
summary_df_woe['rank'] = range(1, len(summary_df_woe)+1)

print(f"\n{'='*100}")
print("RANKING INFORMATION VALUE (IV)")
print(f"{'='*100}")
print(summary_df_woe[['rank','variable','iv','poder_predictivo','n_bins','binning_type']].to_string(index=False))

# Ranking gráfico verde
plot_iv_ranking_verde(summary_df_woe, config.TARGET_COLUMN)

# Gráficos detallados por variable
for i, var in enumerate(summary_df_woe['variable'], 1):
    if var in detalle:
        iv_var = summary_df_woe[summary_df_woe['variable']==var]['iv'].values[0]
        print(f"\n{'─'*100}")
        print(f"[{i}/{len(detalle)}] {var} | IV: {iv_var:.4f} ({clasificar_iv(iv_var)})")
        print(detalle[var].to_string(index=False))
        plot_woe_verde(detalle[var], var, i, len(detalle))

print(f"\n✓ ANÁLISIS WOE COMPLETADO - {len(detalle)}/{len(variables_modelo)} variables")

In [ ]:
vars_seleccionadas =[   'ciiu_val_encoded',
    'tiempo_vida_empresa',
    'flg_pj',
    'ingreso_bruto_total_rrll',
    'prm_sldtotfintrx12m',
    'prm_sldtotfintrx06m',
    'max_sldtotfinpr06m',
    'max_sldtotprmpr06m',
    'max_sldtotfinpr03m',
    'deuda_noibk_max_12m',
    'cant_empresas_max_12m',
    'deuda_total_var_pct_12m',
    'rrll_deuda_total',
    'rrll_cant_entidades',
    'ibk_noizi_mm6',
    'ibk_noizi_mm3',
    'ibk_noizi_max6',
    'ibk_noizi_min3',
    'cre_sldtotfintrxm03',
    'max_saldo_p_u6m',
    'rat_sldtotfinprm12']

In [ ]:
# =============================================================================
# 10. ANÁLISIS BIVARIADO
# =============================================================================

df_bivariado = analisis_bivariado(
    X_train=X_train,
    y_train=y_train,
    vars_analizar=vars_seleccionadas,
    n_bins=10,
    verbose=True
)


# =============================================================================
# 11. ANÁLISIS DE ESTABILIDAD TEMPORAL (PSI)
# =============================================================================

df_psi = analizar_estabilidad_temporal(
    X_train=X_train,
    X_valid=X_valid,
    X_test=X_test,
    vars_analizar=vars_seleccionadas,
    time_col=None,
    verbose=True
)



# =============================================================================
# 12. ANÁLISIS DE ESTABILIDAD POBLACIONAL (CSI)
# =============================================================================

df_csi = analizar_estabilidad_poblacional(
    X_train=X_train,
    y_train=y_train,
    vars_analizar=vars_seleccionadas,
    segmento_col=None,  # O especifica una columna como 'region', 'canal', etc.
    n_bins=10,
    verbose=True
)

